In [76]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LassoCV
import numpy as np
import subprocess
from matplotlib_venn import venn3
import matplotlib.pyplot as plt

In [3]:
expression_data = pd.read_csv("vst_degs.csv")

In [4]:
expression_data.head()

,Unnamed: 0,chp_26,chp_31,chp_34,chp_38,chp_1,chp_3,chp_4,chp_11,chp_5,...,chp_106,chp_108,chp_109,chp_110,chp_111,chp_112,chp_113,chp_114,ipf_1125,ipf_1130
0,TMED7-TICAM2,2.541027,4.593772,4.411618,4.472838,3.411323,3.938280,3.371866,3.069651,4.132273,...,3.396303,2.997156,4.753148,2.925652,1.986403,2.458367,4.443182,4.500799,11.982528,11.948790
1,SNX5,11.664378,11.702990,11.858150,11.659046,11.590148,11.707977,11.733125,11.484413,11.441497,...,11.656847,11.740276,11.855453,11.541765,11.427096,11.610635,11.740794,11.699601,13.199010,13.516239
2,H3F3AP4,3.153035,3.372185,3.238137,2.450154,2.194782,1.734829,2.775714,1.950768,2.636781,...,0.954951,1.203205,4.585984,1.203205,8.062915,1.203205,1.203205,1.203205,17.862013,15.062174
3,AC093010.3,8.824927,8.691420,9.160692,8.637415,8.637578,8.860465,9.265420,8.937777,9.035598,...,8.597248,9.226443,9.120908,7.394812,8.768886,8.094580,9.682386,9.673836,15.016835,15.234279
4,HERC3,9.972369,10.291667,10.572234,9.945113,9.723398,10.244080,10.051501,10.134745,10.332995,...,9.749555,10.776304,11.055233,9.193449,10.619217,9.284438,10.497540,10.062599,13.940517,13.785838


In [5]:
print(expression_data.shape)

(15744, 289)


In [6]:
print("First 5 columns:", expression_data.columns[:5].tolist())

First 5 columns: ['Unnamed: 0', 'chp_26', 'chp_31', 'chp_34', 'chp_38']


In [7]:
expression = expression_data.set_index('Unnamed: 0')

In [8]:
print("First 5 column names:", expression_data.columns[:5].tolist())

First 5 column names: ['Unnamed: 0', 'chp_26', 'chp_31', 'chp_34', 'chp_38']


In [9]:
print(expression_data.shape)

(15744, 289)


In [10]:
metadata = pd.read_csv("meta_final.csv")

In [11]:
print(metadata.shape)

(288, 6)


In [12]:
print(metadata['diagnosis'].value_counts())

diagnosis
ipf        103
control    103
chp         82
Name: count, dtype: int64


In [13]:
X = expression.T #need to transpose because rows should samples and column should genes

In [14]:
y = metadata['diagnosis']

In [15]:
print("X index (first 5):", X.index[:5].tolist())
print("y index (first 5):", y.index[:5].tolist())

X index (first 5): ['chp_26', 'chp_31', 'chp_34', 'chp_38', 'chp_1']
y index (first 5): [0, 1, 2, 3, 4]


In [16]:
metadata = metadata.set_index('sample_id')

In [17]:
y = metadata['diagnosis']

In [18]:
X = X.loc[y.index]

In [19]:
lab = LabelEncoder()
y_encoded = lab.fit_transform(y)

In [20]:
print("Label encoding:")
for label, code in zip(lab.classes_, range(len(lab.classes_))):
    print(f"  {label} : {code}")

Label encoding:
  chp : 0
  control : 1
  ipf : 2


In [21]:
X_scaled = StandardScaler().fit_transform(X)

In [22]:
X_scaled = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)

In [23]:
print("Scaled data shape:", X_scaled.shape)

Scaled data shape: (288, 15744)


**Feature Selection Method 01 : SVM RFE**

In [24]:
#Define SVM classifier
svm = LinearSVC(max_iter=2000, random_state=42)

In [25]:
rfe = RFE(estimator=svm,n_features_to_select=1000,step=100)

In [26]:
rfe.fit(X_scaled, y_encoded)

,estimator,LinearSVC(max...ndom_state=42)
,n_features_to_select,1000
,step,100
,verbose,0
,importance_getter,'auto'
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'


In [27]:
selected_svm = X_scaled.columns[rfe.support_].tolist()

In [28]:
selected_svm_df = pd.DataFrame({'gene': selected_svm })

In [32]:
selected_svm_df.to_csv( "datasets/selected_svm.csv",index=False)

**Feature Selection Method 02 : DECISION TREES**

In [29]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_scaled, y_encoded)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [30]:
dt_importances = pd.Series(dt_model.feature_importances_,index=X_scaled.columns)

In [31]:
dt_top_genes = dt_importances.nlargest(1000).index.tolist()

In [34]:
print("Number of genes selected from decision tree:", len(dt_top_genes))

Number of genes selected from decision tree: 1000


In [35]:
print("First 10 genes:", dt_top_genes[:10])

First 10 genes: ['SCARB2', 'COL17A1', 'RNF182', 'SYNJ2', 'SAPCD2', 'PAK2', 'TMED7-TICAM2', 'SNX5', 'H3F3AP4', 'AC093010.3']


In [36]:
if 'ITGAV' in dt_top_genes:
    rank = dt_importances.rank(ascending=False)[['ITGAV']].values[0]
    print(f"\nITGAV FOUND! Ranked #{int(rank)} out of 15,744 genes")
else:
    print("\nITGAV not in top 1000 for Decision Trees")


ITGAV not in top 1000 for Decision Trees


In [37]:
dt_df = pd.DataFrame({'gene': dt_top_genes})

In [38]:
dt_df.to_csv("datasets/decision_tree_genes.csv",index=False)

**Feature Selection Method 03 : LASSO Regression**

In [41]:
lasso = LassoCV(cv=5, random_state=42, max_iter=5000)

In [43]:
#LASSO only works with binary classification
# So we create a binary label: CHP=1, others=0
# This finds genes most important for identifying CHP
y_binary = (y == 'chp').astype(int)

In [44]:
lasso.fit(X_scaled, y_binary)

,eps,0.001
,n_alphas,'deprecated'
,alphas,'warn'
,fit_intercept,True
,precompute,'auto'
,max_iter,5000
,tol,0.0001
,copy_X,True
,cv,5
,verbose,False
,n_jobs,None


In [46]:
# Step 4 - Get genes with non-zero coefficients
# LASSO sets unimportant genes to exactly 0
lasso_coef = pd.Series(lasso.coef_, index=X_scaled.columns)
lasso_genes = lasso_coef[lasso_coef != 0].index.tolist()

In [47]:
print("Best penalty (alpha):", round(lasso.alpha_, 6))

Best penalty (alpha): 0.000442


In [48]:
print("Number of genes selected from LASSO:", len(lasso_genes))

Number of genes selected from LASSO: 282


In [49]:
print("First 10 genes:", lasso_genes[:10])

First 10 genes: ['H3F3AP4', 'AC024558.1', 'UBE2Q2P6', 'LINC00211', 'SCARNA6', 'AL353608.3', 'ATOH7', 'AC068888.2', 'CCNE2', 'LINC02002']


In [50]:
if 'ITGAV' in lasso_genes:
    print("\nITGAV FOUND in LASSO!")
else:
    print("\nITGAV not found in LASSO")


ITGAV not found in LASSO


In [51]:
lasso_df = pd.DataFrame({'gene': lasso_genes})
lasso_df.to_csv("datasets/lasso_genes.csv",index=False)

**Total genes selected from each Feature selection method**

In [52]:
svm_set = set(selected_svm)  
dt_set  = set(dt_top_genes)  
lasso_set = set(lasso_genes)   

In [56]:
print("Number of genes selected from SVM:", len(svm_set))
print("Number of genes selected from decision tree:", len(dt_set))
print("Number of genes selected from lasso:", len(lasso_set))

Number of genes selected from SVM: 1000
Number of genes selected from decision tree: 1000
Number of genes selected from lasso: 282


In [57]:
# Find overlap between all 3 methods
overlap_genes = svm_set & dt_set & lasso_set

In [58]:
overlap_svm_dt    = svm_set & dt_set
overlap_svm_lasso = svm_set & lasso_set
overlap_dt_lasso  = dt_set  & lasso_set

In [61]:
print(f"SVM-RFE & DT overlap:   {len(overlap_svm_dt)}")
print(f"SVM-RFE & LASSO overlap:{len(overlap_svm_lasso)}")
print(f"DT & LASSO overlap:     {len(overlap_dt_lasso)}")

SVM-RFE & DT overlap:   2
SVM-RFE & LASSO overlap:28
DT & LASSO overlap:     55


In [63]:
overlap_all = svm_set & dt_set & lasso_set

In [64]:
print(f"ALL 3 methods overlap:  {len(overlap_all)}")

ALL 3 methods overlap:  0


In [66]:
final_features = list(overlap_all)

In [67]:
final_df = pd.DataFrame({'gene': final_features})
final_df.to_csv("datasets/final_features.csv",index=False)

In [68]:
# Union of all pairwise overlaps
final_features = (overlap_svm_dt | overlap_svm_lasso | overlap_dt_lasso)

In [69]:
print("UPDATED FEATURE SELECTION SUMMARY")
print(f"SVM-RFE & DT overlap:      {len(overlap_svm_dt)}")
print(f"SVM-RFE & LASSO overlap:   {len(overlap_svm_lasso)}")
print(f"DT & LASSO overlap:        {len(overlap_dt_lasso)}")
print(f"Final features (>=2 methods): {len(final_features)}")

UPDATED FEATURE SELECTION SUMMARY
SVM-RFE & DT overlap:      2
SVM-RFE & LASSO overlap:   28
DT & LASSO overlap:        55
Final features (>=2 methods): 85


In [70]:
if 'ITGAV' in final_features:
    print("ITGAV FOUND in final feature set! ")
else:
    print("ITGAV not in final feature set")

ITGAV not in final feature set


In [71]:
final_features_list = list(final_features)

In [72]:
final_df = pd.DataFrame({'gene': final_features_list})
final_df.to_csv("datasets/final_features.csv",index=False)

In [73]:
X_final = X[final_features_list]
print("Final expression matrix shape:", X_final.shape)

Final expression matrix shape: (288, 85)


**venn diagram**

In [ ]:

# VENN DIAGRAM — FEATURE SELECTION VISUALIZATION


# Install matplotlib-venn
import subprocess
subprocess.run(['pip', 'install', 'matplotlib-venn'], 
               capture_output=True)

from matplotlib_venn import venn3
import matplotlib.pyplot as plt

# Create Venn diagram
plt.figure(figsize=(10, 7))

venn3(subsets  = [svm_set, dt_set, lasso_set],
      set_labels = ('SVM-RFE\n(1000)', 
                    'Decision Trees\n(1000)', 
                    'LASSO\n(282)'),
      set_colors = ('#E74C3C', '#3498DB', '#2ECC71'),
      alpha      = 0.6)

plt.title("Feature Selection — Gene Overlap Across Methods\n"
          f"Final Feature Set: 85 genes (selected by ≥2 methods)",
          fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(
    "D:/Research/implementation/step02/venn_diagram.png",
    dpi=300, bbox_inches='tight')
plt.show()
print("Venn diagram saved!")